# MB52 | Refinamento do achado de cobertura histórica | Centro 4128

**Objetivo:** testar se as **139.880 chaves Material + Depósito presentes no SAP e não localizadas no Datalake** representam perda histórica real, diferença de escopo, filtro de saldo zero, snapshot incorreto ou problema de normalização/chave.

## Antes de executar
Faça uma única substituição no notebook:
- `<TABELA_DATALAKE>` pelo nome completo da tabela do Datalake.
- `<TABELA_SAP_STAGING>` por uma tabela temporária/staging contendo o extrato MB52 usado na comparação.

Colunas esperadas na staging SAP: `Material`, `Deposito`, `Utilizacao_livre`, `Val_utiliz_livre`.

> Se os nomes diferirem, ajuste somente a CTE `sap_base` da Query 04. As demais queries reutilizam a mesma lógica.


In [0]:
SELECT current_catalog(), current_schema();

In [0]:
-- QUERY 02 | Verificar se o centro 4128 aparece em todas as partições técnicas
SELECT
    yearingest,
    monthingest,
    CAST(dateingest AS DATE) AS data_snapshot,
    COUNT(*) AS linhas,
    COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '')) AS materiais,
    COUNT(DISTINCT UPPER(TRIM(CAST(cod_deposito AS STRING)))) AS depositos
FROM <TABELA_DATALAKE>
WHERE TRIM(CAST(cod_centro AS STRING)) = '4128'
GROUP BY yearingest, monthingest, CAST(dateingest AS DATE)
ORDER BY data_snapshot, yearingest, monthingest;

In [0]:
-- QUERY 03 | Perfil por depósito no snapshot mais recente
-- Responde: depósitos inteiros ficaram fora ou com volume anormalmente baixo?
WITH ultima_data AS (
    SELECT MAX(CAST(dateingest AS DATE)) AS dt
    FROM <TABELA_DATALAKE>
    WHERE TRIM(CAST(cod_centro AS STRING)) = '4128'
)
SELECT
    UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito,
    COUNT(*) AS linhas,
    COUNT(DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '')) AS materiais,
    SUM(CASE WHEN COALESCE(CAST(qt_utilizacao_livre AS DECIMAL(38,6)),0) <> 0 THEN 1 ELSE 0 END) AS linhas_qtd_nao_zero,
    SUM(CASE WHEN COALESCE(CAST(vl_utilizacao_livre AS DECIMAL(38,6)),0) <> 0 THEN 1 ELSE 0 END) AS linhas_valor_nao_zero
FROM <TABELA_DATALAKE> d
CROSS JOIN ultima_data u
WHERE TRIM(CAST(d.cod_centro AS STRING)) = '4128'
  AND CAST(d.dateingest AS DATE) = u.dt
GROUP BY UPPER(TRIM(CAST(cod_deposito AS STRING)))
ORDER BY materiais DESC;

In [0]:
-- QUERY 04 | Construir bases normalizadas e medir o anti-join SAP -> Datalake
-- IMPORTANTE: execute esta query como um bloco. Ela recalcula o número de 139.880.
WITH sap_base AS (
    SELECT
        REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') AS material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) AS deposito_norm,
        CAST(Utilizacao_livre AS DECIMAL(38,6)) AS qtd_sap,
        CAST(Val_utiliz_livre AS DECIMAL(38,6)) AS valor_sap
    FROM <TABELA_SAP_STAGING>
    WHERE Material IS NOT NULL
),
ultima_data AS (
    SELECT MAX(CAST(dateingest AS DATE)) AS dt
    FROM <TABELA_DATALAKE>
    WHERE TRIM(CAST(cod_centro AS STRING)) = '4128'
),
dl_base AS (
    SELECT DISTINCT
        REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') AS material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito_norm
    FROM <TABELA_DATALAKE> d
    CROSS JOIN ultima_data u
    WHERE TRIM(CAST(d.cod_centro AS STRING)) = '4128'
      AND CAST(d.dateingest AS DATE) = u.dt
)
SELECT
    COUNT(DISTINCT CONCAT(s.material_norm,'|',s.deposito_norm)) AS chaves_sap,
    COUNT(DISTINCT CASE WHEN d.material_norm IS NOT NULL THEN CONCAT(s.material_norm,'|',s.deposito_norm) END) AS chaves_comuns,
    COUNT(DISTINCT CASE WHEN d.material_norm IS NULL THEN CONCAT(s.material_norm,'|',s.deposito_norm) END) AS chaves_somente_sap
FROM sap_base s
LEFT JOIN dl_base d
  ON s.material_norm = d.material_norm
 AND s.deposito_norm = d.deposito_norm;

In [0]:
-- QUERY 05 | Separar as ausências por saldo/valor no SAP
-- Teste central: ausência histórica real versus exclusão de registros zerados.
WITH sap_base AS (
    SELECT
        REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') AS material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) AS deposito_norm,
        COALESCE(CAST(Utilizacao_livre AS DECIMAL(38,6)),0) AS qtd_sap,
        COALESCE(CAST(Val_utiliz_livre AS DECIMAL(38,6)),0) AS valor_sap
    FROM <TABELA_SAP_STAGING>
    WHERE Material IS NOT NULL
), ultima_data AS (
    SELECT MAX(CAST(dateingest AS DATE)) AS dt FROM <TABELA_DATALAKE>
    WHERE TRIM(CAST(cod_centro AS STRING)) = '4128'
), dl_base AS (
    SELECT DISTINCT
        REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') AS material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) AS deposito_norm
    FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
    WHERE TRIM(CAST(d.cod_centro AS STRING)) = '4128' AND CAST(d.dateingest AS DATE)=u.dt
), ausentes AS (
    SELECT s.* FROM sap_base s LEFT ANTI JOIN dl_base d
      ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
)
SELECT
    CASE
      WHEN qtd_sap = 0 AND valor_sap = 0 THEN 'QTD_E_VALOR_ZERO'
      WHEN qtd_sap <> 0 AND valor_sap = 0 THEN 'QTD_NAO_ZERO_VALOR_ZERO'
      WHEN qtd_sap = 0 AND valor_sap <> 0 THEN 'QTD_ZERO_VALOR_NAO_ZERO'
      ELSE 'QTD_E_VALOR_NAO_ZERO'
    END AS classe_saldo,
    COUNT(*) AS linhas,
    COUNT(DISTINCT CONCAT(material_norm,'|',deposito_norm)) AS chaves,
    COUNT(DISTINCT material_norm) AS materiais,
    COUNT(DISTINCT deposito_norm) AS depositos
FROM ausentes
GROUP BY 1
ORDER BY chaves DESC;

In [0]:
-- QUERY 06 | Ausências por depósito e classe de saldo
-- Objetivo: localizar concentração estrutural em depósitos específicos.
WITH sap_base AS (
    SELECT REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') material_norm,
           UPPER(TRIM(CAST(Deposito AS STRING))) deposito_norm,
           COALESCE(CAST(Utilizacao_livre AS DECIMAL(38,6)),0) qtd_sap,
           COALESCE(CAST(Val_utiliz_livre AS DECIMAL(38,6)),0) valor_sap
    FROM <TABELA_SAP_STAGING> WHERE Material IS NOT NULL
), ultima_data AS (
    SELECT MAX(CAST(dateingest AS DATE)) dt FROM <TABELA_DATALAKE>
    WHERE TRIM(CAST(cod_centro AS STRING))='4128'
), dl_base AS (
    SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
           UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm
    FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
    WHERE TRIM(CAST(d.cod_centro AS STRING))='4128' AND CAST(d.dateingest AS DATE)=u.dt
), ausentes AS (
    SELECT s.* FROM sap_base s LEFT ANTI JOIN dl_base d
      ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
)
SELECT deposito_norm,
       CASE WHEN qtd_sap=0 AND valor_sap=0 THEN 'ZERO' ELSE 'NAO_ZERO' END classe_saldo,
       COUNT(DISTINCT CONCAT(material_norm,'|',deposito_norm)) chaves,
       COUNT(DISTINCT material_norm) materiais,
       SUM(qtd_sap) soma_quantidade,
       SUM(valor_sap) soma_valor
FROM ausentes
GROUP BY deposito_norm, CASE WHEN qtd_sap=0 AND valor_sap=0 THEN 'ZERO' ELSE 'NAO_ZERO' END
ORDER BY chaves DESC;

In [0]:
-- QUERY 07 | Materiais ausentes em um depósito, mas presentes em outro no Datalake
-- Indica possível perda da dimensão Depósito ou regra de carga por depósito.
WITH sap_base AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) deposito_norm
 FROM <TABELA_SAP_STAGING> WHERE Material IS NOT NULL
), ultima_data AS (
 SELECT MAX(CAST(dateingest AS DATE)) dt FROM <TABELA_DATALAKE>
 WHERE TRIM(CAST(cod_centro AS STRING))='4128'
), dl_base AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm
 FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
 WHERE TRIM(CAST(d.cod_centro AS STRING))='4128' AND CAST(d.dateingest AS DATE)=u.dt
), ausentes AS (
 SELECT s.* FROM sap_base s LEFT ANTI JOIN dl_base d
 ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
)
SELECT
  COUNT(*) AS chaves_ausentes,
  SUM(CASE WHEN EXISTS (SELECT 1 FROM dl_base x WHERE x.material_norm=a.material_norm) THEN 1 ELSE 0 END) AS material_existe_em_outro_deposito_dl,
  SUM(CASE WHEN NOT EXISTS (SELECT 1 FROM dl_base x WHERE x.material_norm=a.material_norm) THEN 1 ELSE 0 END) AS material_nao_existe_no_dl
FROM ausentes a;

In [0]:
-- QUERY 08 | Persistência histórica das chaves que estão ausentes no snapshot mais recente
-- Responde: a chave nunca existiu no DL ou existiu em snapshot anterior?
WITH sap_base AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) deposito_norm
 FROM <TABELA_SAP_STAGING> WHERE Material IS NOT NULL
), ultima_data AS (
 SELECT MAX(CAST(dateingest AS DATE)) dt FROM <TABELA_DATALAKE>
 WHERE TRIM(CAST(cod_centro AS STRING))='4128'
), dl_ultimo AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm
 FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
 WHERE TRIM(CAST(d.cod_centro AS STRING))='4128' AND CAST(d.dateingest AS DATE)=u.dt
), ausentes AS (
 SELECT s.* FROM sap_base s LEFT ANTI JOIN dl_ultimo d
 ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
), dl_historico AS (
 SELECT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm,
        MIN(CAST(dateingest AS DATE)) primeira_data,
        MAX(CAST(dateingest AS DATE)) ultima_data,
        COUNT(DISTINCT CAST(dateingest AS DATE)) qtd_snapshots
 FROM <TABELA_DATALAKE>
 WHERE TRIM(CAST(cod_centro AS STRING))='4128'
 GROUP BY 1,2
)
SELECT
 CASE WHEN h.material_norm IS NULL THEN 'NUNCA_EXISTIU_NO_DL'
      ELSE 'EXISTIU_EM_SNAPSHOT_ANTERIOR' END AS situacao,
 COUNT(*) AS chaves,
 MIN(h.primeira_data) AS menor_primeira_data,
 MAX(h.ultima_data) AS maior_ultima_data,
 MAX(h.qtd_snapshots) AS max_snapshots_por_chave
FROM ausentes a
LEFT JOIN dl_historico h ON a.material_norm=h.material_norm AND a.deposito_norm=h.deposito_norm
GROUP BY 1;

In [0]:
-- QUERY 09 | Testar problemas de normalização na chave bruta
-- Procura espaços, zeros à esquerda, depósito vazio e material que vira vazio após normalização.
SELECT
  SUM(CASE WHEN CAST(cod_material AS STRING) <> TRIM(CAST(cod_material AS STRING)) THEN 1 ELSE 0 END) AS material_com_espacos,
  SUM(CASE WHEN TRIM(CAST(cod_material AS STRING)) RLIKE '^0+' THEN 1 ELSE 0 END) AS material_com_zero_esquerda,
  SUM(CASE WHEN cod_deposito IS NULL OR TRIM(CAST(cod_deposito AS STRING))='' THEN 1 ELSE 0 END) AS deposito_vazio,
  SUM(CASE WHEN CAST(cod_deposito AS STRING) <> TRIM(CAST(cod_deposito AS STRING)) THEN 1 ELSE 0 END) AS deposito_com_espacos,
  SUM(CASE WHEN REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '')='' THEN 1 ELSE 0 END) AS material_vazio_apos_normalizacao
FROM <TABELA_DATALAKE>
WHERE TRIM(CAST(cod_centro AS STRING))='4128';

In [0]:
-- QUERY 10 | Unicidade Material + Depósito dentro de cada snapshot
-- Distingue duplicata real de repetição entre snapshots.
SELECT
  CAST(dateingest AS DATE) AS data_snapshot,
  COUNT(*) AS linhas,
  COUNT(DISTINCT CONCAT(REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', ''),'|',UPPER(TRIM(CAST(cod_deposito AS STRING))))) AS chaves,
  COUNT(*) - COUNT(DISTINCT CONCAT(REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', ''),'|',UPPER(TRIM(CAST(cod_deposito AS STRING))))) AS excedente_sobre_chave
FROM <TABELA_DATALAKE>
WHERE TRIM(CAST(cod_centro AS STRING))='4128'
GROUP BY CAST(dateingest AS DATE)
ORDER BY data_snapshot;

In [0]:
-- QUERY 11 | Exemplos das ausências com saldo/valor não zero
-- Retorna casos concretos prioritários para investigação do pipeline.
WITH sap_base AS (
 SELECT REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) deposito_norm,
        COALESCE(CAST(Utilizacao_livre AS DECIMAL(38,6)),0) qtd_sap,
        COALESCE(CAST(Val_utiliz_livre AS DECIMAL(38,6)),0) valor_sap
 FROM <TABELA_SAP_STAGING> WHERE Material IS NOT NULL
), ultima_data AS (
 SELECT MAX(CAST(dateingest AS DATE)) dt FROM <TABELA_DATALAKE>
 WHERE TRIM(CAST(cod_centro AS STRING))='4128'
), dl_base AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm
 FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
 WHERE TRIM(CAST(d.cod_centro AS STRING))='4128' AND CAST(d.dateingest AS DATE)=u.dt
)
SELECT s.material_norm AS material, s.deposito_norm AS deposito, s.qtd_sap, s.valor_sap
FROM sap_base s LEFT ANTI JOIN dl_base d
 ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
WHERE s.qtd_sap<>0 OR s.valor_sap<>0
ORDER BY ABS(s.valor_sap) DESC, ABS(s.qtd_sap) DESC
LIMIT 100;

In [0]:
-- QUERY 12 | Resumo final para decisão do achado
-- Copie o resultado desta query junto com os resultados das Queries 01, 05, 06, 08 e 11.
WITH sap_base AS (
 SELECT REGEXP_REPLACE(TRIM(CAST(Material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(Deposito AS STRING))) deposito_norm,
        COALESCE(CAST(Utilizacao_livre AS DECIMAL(38,6)),0) qtd_sap,
        COALESCE(CAST(Val_utiliz_livre AS DECIMAL(38,6)),0) valor_sap
 FROM <TABELA_SAP_STAGING> WHERE Material IS NOT NULL
), ultima_data AS (
 SELECT MAX(CAST(dateingest AS DATE)) dt FROM <TABELA_DATALAKE>
 WHERE TRIM(CAST(cod_centro AS STRING))='4128'
), dl_base AS (
 SELECT DISTINCT REGEXP_REPLACE(TRIM(CAST(cod_material AS STRING)), '^0+', '') material_norm,
        UPPER(TRIM(CAST(cod_deposito AS STRING))) deposito_norm
 FROM <TABELA_DATALAKE> d CROSS JOIN ultima_data u
 WHERE TRIM(CAST(d.cod_centro AS STRING))='4128' AND CAST(d.dateingest AS DATE)=u.dt
), ausentes AS (
 SELECT s.* FROM sap_base s LEFT ANTI JOIN dl_base d
 ON s.material_norm=d.material_norm AND s.deposito_norm=d.deposito_norm
)
SELECT
 (SELECT dt FROM ultima_data) AS snapshot_dl_usado,
 COUNT(DISTINCT CONCAT(material_norm,'|',deposito_norm)) AS chaves_ausentes_total,
 COUNT(DISTINCT CASE WHEN qtd_sap=0 AND valor_sap=0 THEN CONCAT(material_norm,'|',deposito_norm) END) AS chaves_ausentes_zeradas,
 COUNT(DISTINCT CASE WHEN qtd_sap<>0 OR valor_sap<>0 THEN CONCAT(material_norm,'|',deposito_norm) END) AS chaves_ausentes_com_saldo_ou_valor,
 COUNT(DISTINCT material_norm) AS materiais_afetados,
 COUNT(DISTINCT deposito_norm) AS depositos_afetados,
 SUM(qtd_sap) AS quantidade_sap_ausente,
 SUM(valor_sap) AS valor_sap_ausente
FROM ausentes;

## Resultados que preciso receber
Envie as saídas das **Queries 01, 05, 06, 08, 11 e 12**.

## Regra de interpretação
- Se a maior parte das ausências estiver zerada, isso **não elimina automaticamente** o problema. É preciso confirmar se a tabela foi desenhada para estoque atual ou histórico.
- Chaves com quantidade ou valor não zero são o subconjunto prioritário e sustentam perda funcional mesmo que exista uma regra de descarte de saldos zerados.
- Se as chaves existirem em snapshots anteriores, o problema pode ser retenção/snapshot e não ausência histórica absoluta.
- Se nunca existiram, a causa permanece não identificada até testar filtros e regras do pipeline.
